# 04 — Results and Quality Control

This notebook turns the raw outputs from notebook 03 into a concise results package:

- tissue volume summary tables
- QC plots for GM / WM / CSF segmentation
- a portfolio-ready summary figure
- a small metrics CSV for reproducible reporting

**Required inputs**
- `data/processed/brain_stripped.nii.gz`
- `data/processed/brain_stripped_mask.nii.gz`
- `outputs/segmentations/gm_probability.nii.gz`
- `outputs/segmentations/wm_probability.nii.gz`
- `outputs/segmentations/csf_probability.nii.gz`
- `outputs/segmentations/tissue_segmentation_3class.nii.gz`
- `outputs/segmentations/tissue_volumes.csv`

**Outputs created here**
- `outputs/figures/04_tissue_volume_breakdown.png`
- `outputs/figures/04_segmentation_qc.png`
- `outputs/figures/04_portfolio_summary.png`
- `outputs/segmentations/tissue_qc_metrics.csv`

In [ ]:
import numpy as np
import nibabel as nib
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path

In [ ]:
# ── Paths ─────────────────────────────────
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "data").exists() and (CWD / "outputs").exists() else CWD.parent
assert (ROOT / "data").exists(), f"Could not locate project root from {CWD}"
PROC = ROOT / "data" / "processed"
SEG = ROOT / "outputs" / "segmentations"
FIG = ROOT / "outputs" / "figures"

FIG.mkdir(parents=True, exist_ok=True)

BRAIN_PATH = PROC / "brain_stripped.nii.gz"
MASK_PATH = PROC / "brain_stripped_mask.nii.gz"
GM_PATH = SEG / "gm_probability.nii.gz"
WM_PATH = SEG / "wm_probability.nii.gz"
CSF_PATH = SEG / "csf_probability.nii.gz"
HARD_PATH = SEG / "tissue_segmentation_3class.nii.gz"
VOLUME_CSV = SEG / "tissue_volumes.csv"
QC_CSV = SEG / "tissue_qc_metrics.csv"

FIG_VOLUME = FIG / "04_tissue_volume_breakdown.png"
FIG_QC = FIG / "04_segmentation_qc.png"
FIG_PORTFOLIO = FIG / "04_portfolio_summary.png"

In [ ]:
# ── Helpers ────────────────────────────────
TISSUE_COLORS = {
    "GM": "#d95f02",
    "WM": "#1b9e77",
    "CSF": "#7570b3",
}
SEG_CMAP = ListedColormap([TISSUE_COLORS["GM"], TISSUE_COLORS["WM"], TISSUE_COLORS["CSF"]])
LEGEND_HANDLES = [
    Patch(color=TISSUE_COLORS["GM"], label="GM"),
    Patch(color=TISSUE_COLORS["WM"], label="WM"),
    Patch(color=TISSUE_COLORS["CSF"], label="CSF"),
]


def load_float_data(path):
    return np.asarray(nib.load(str(path)).get_fdata(), dtype=np.float32)


def rotate_mid_slices(volume):
    x_mid, y_mid, z_mid = [s // 2 for s in volume.shape]
    return {
        "Sagittal": np.rot90(volume[x_mid, :, :]),
        "Coronal": np.rot90(volume[:, y_mid, :]),
        "Axial": np.rot90(volume[:, :, z_mid]),
    }


def overlay_segmentation(base, labels, alpha=0.45):
    masked = np.ma.masked_where(labels == 0, labels)
    return base, masked, alpha


def save_show(fig, path):
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()

## Step 1 — Load segmentation outputs

In [ ]:
for path in [BRAIN_PATH, MASK_PATH, GM_PATH, WM_PATH, CSF_PATH, HARD_PATH, VOLUME_CSV]:
    assert path.exists(), f"Missing required file: {path}"

brain_img = nib.load(str(BRAIN_PATH))
brain = np.asarray(brain_img.get_fdata(), dtype=np.float32)
mask = load_float_data(MASK_PATH) > 0.5
gm = load_float_data(GM_PATH)
wm = load_float_data(WM_PATH)
csf = load_float_data(CSF_PATH)
hard = np.asarray(nib.load(str(HARD_PATH)).get_fdata(), dtype=np.uint8)
volume_table = pd.read_csv(VOLUME_CSV)

print(f"Brain shape : {brain.shape}")
print(f"Voxel size  : {brain_img.header.get_zooms()[:3]}")
print(f"Unique hard labels: {np.unique(hard)}")
volume_table

## Step 2 — Derive reporting metrics

Alongside the model's saved tissue volumes, we compute a few simple QC metrics:
- mean and spread of `GM + WM + CSF` probabilities inside the brain mask
- voxel counts per hard-label class
- brain tissue volume (`GM + WM`)
- GM/WM ratio

In [ ]:
prob_sum = gm + wm + csf
prob_sum_in_mask = prob_sum[mask]

gm_vol = float(volume_table.loc[volume_table["tissue"] == "GM", "volume_cm3"].iloc[0])
wm_vol = float(volume_table.loc[volume_table["tissue"] == "WM", "volume_cm3"].iloc[0])
csf_vol = float(volume_table.loc[volume_table["tissue"] == "CSF", "volume_cm3"].iloc[0])
tiv = float(volume_table.loc[volume_table["tissue"] == "TIV", "volume_cm3"].iloc[0])

brain_tissue_vol = gm_vol + wm_vol
gm_wm_ratio = gm_vol / wm_vol

hard_voxel_counts = {
    "gm_voxels": int((hard == 1).sum()),
    "wm_voxels": int((hard == 2).sum()),
    "csf_voxels": int((hard == 3).sum()),
}

qc_metrics = pd.DataFrame(
    {
        "metric": [
            "gm_volume_cm3",
            "wm_volume_cm3",
            "csf_volume_cm3",
            "tiv_cm3",
            "brain_tissue_volume_cm3",
            "gm_wm_ratio",
            "probability_sum_mean_in_mask",
            "probability_sum_std_in_mask",
            "probability_sum_min_in_mask",
            "probability_sum_max_in_mask",
            "gm_voxels_hard",
            "wm_voxels_hard",
            "csf_voxels_hard",
        ],
        "value": [
            gm_vol,
            wm_vol,
            csf_vol,
            tiv,
            brain_tissue_vol,
            gm_wm_ratio,
            float(prob_sum_in_mask.mean()),
            float(prob_sum_in_mask.std()),
            float(prob_sum_in_mask.min()),
            float(prob_sum_in_mask.max()),
            hard_voxel_counts["gm_voxels"],
            hard_voxel_counts["wm_voxels"],
            hard_voxel_counts["csf_voxels"],
        ],
    }
)
qc_metrics.to_csv(QC_CSV, index=False)

summary_table = volume_table.copy()
summary_table["percent_of_tiv"] = 100 * summary_table["fraction_of_tiv"]
summary_table

## Step 3 — Volume breakdown figure

In [ ]:
plot_table = volume_table[volume_table["tissue"] != "TIV"].copy()
plot_table["color"] = plot_table["tissue"].map(TISSUE_COLORS)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(
    plot_table["tissue"],
    plot_table["volume_cm3"],
    color=plot_table["color"],
    edgecolor="black",
)
axes[0].set_title("Absolute Tissue Volumes")
axes[0].set_ylabel("Volume (cm^3)")
axes[0].grid(axis="y", alpha=0.25)

for i, value in enumerate(plot_table["volume_cm3"]):
    axes[0].text(i, value + 15, f"{value:.1f}", ha="center", va="bottom", fontsize=10)

axes[1].pie(
    plot_table["fraction_of_tiv"],
    labels=plot_table["tissue"],
    autopct="%1.1f%%",
    colors=plot_table["color"],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 1.0},
)
axes[1].set_title("Relative Tissue Composition")

fig.suptitle("Brain Tissue Volume Breakdown", fontsize=14)
plt.tight_layout()
save_show(fig, FIG_VOLUME)

## Step 4 — Segmentation QC montage

We inspect the middle slice in all three anatomical planes, overlaying the hard segmentation on top of the skull-stripped brain.

In [ ]:
brain_slices = rotate_mid_slices(brain)
hard_slices = rotate_mid_slices(hard)
gm_slices = rotate_mid_slices(gm)
wm_slices = rotate_mid_slices(wm)
csf_slices = rotate_mid_slices(csf)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
planes = list(brain_slices.keys())

for col, plane in enumerate(planes):
    base, masked, alpha = overlay_segmentation(brain_slices[plane], hard_slices[plane])
    axes[0, col].imshow(base, cmap="gray")
    axes[0, col].imshow(masked, cmap=SEG_CMAP, alpha=alpha, interpolation="none", vmin=1, vmax=3)
    axes[0, col].set_title(f"{plane} overlay")
    axes[0, col].axis("off")

    prob_rgb = np.stack([
        gm_slices[plane],
        wm_slices[plane],
        csf_slices[plane],
    ], axis=-1)
    prob_rgb = np.clip(prob_rgb, 0, 1)
    axes[1, col].imshow(prob_rgb)
    axes[1, col].set_title(f"{plane} GM / WM / CSF probabilities")
    axes[1, col].axis("off")

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Segmentation Quality Control", fontsize=15)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
save_show(fig, FIG_QC)

## Step 5 — Portfolio summary figure

This composes one figure that is easy to drop into a README, case study, or portfolio write-up.

In [ ]:
z_mid = brain.shape[2] // 2
base_axial = np.rot90(brain[:, :, z_mid])
hard_axial = np.rot90(hard[:, :, z_mid])
prob_sum_axial = np.rot90(prob_sum[:, :, z_mid])

fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 2, height_ratios=[1.1, 1.0], width_ratios=[1.05, 1.0])

ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(base_axial, cmap="gray")
ax1.set_title("Skull-stripped MRI (axial)")
ax1.axis("off")

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(base_axial, cmap="gray")
ax2.imshow(np.ma.masked_where(hard_axial == 0, hard_axial), cmap=SEG_CMAP, alpha=0.45, interpolation="none", vmin=1, vmax=3)
ax2.set_title("Tissue segmentation overlay")
ax2.axis("off")

ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(plot_table["tissue"], plot_table["volume_cm3"], color=plot_table["color"], edgecolor="black")
ax3.set_title("Tissue volumes")
ax3.set_ylabel("Volume (cm^3)")
ax3.grid(axis="y", alpha=0.25)

ax4 = fig.add_subplot(gs[1, 1])
ax4.axis("off")
summary_lines = [
    "Automated 3D brain tissue segmentation",
    "",
    f"GM volume   : {gm_vol:.1f} cm^3 ({100 * gm_vol / tiv:.1f}%)",
    f"WM volume   : {wm_vol:.1f} cm^3 ({100 * wm_vol / tiv:.1f}%)",
    f"CSF volume  : {csf_vol:.1f} cm^3 ({100 * csf_vol / tiv:.1f}%)",
    f"TIV         : {tiv:.1f} cm^3",
    f"GM/WM ratio : {gm_wm_ratio:.3f}",
    "",
    f"Prob. sum in mask: {prob_sum_in_mask.mean():.3f} ± {prob_sum_in_mask.std():.3f}",
    f"Hard labels present: {np.unique(hard).astype(int).tolist()}",
]
ax4.text(
    0.02,
    0.98,
    "\n".join(summary_lines),
    va="top",
    ha="left",
    fontsize=12,
    family="monospace",
    bbox={"boxstyle": "round,pad=0.6", "facecolor": "#f7f7f7", "edgecolor": "#cccccc"},
)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=3, frameon=False)
fig.suptitle("BrainScan Portfolio Summary", fontsize=18)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
save_show(fig, FIG_PORTFOLIO)

## Step 6 — Final sanity checks

In [ ]:
print("=" * 60)
print("  RESULTS AND QC SUMMARY")
print("=" * 60)
print(f"GM range        : {gm.min():.4f} -> {gm.max():.4f}")
print(f"WM range        : {wm.min():.4f} -> {wm.max():.4f}")
print(f"CSF range       : {csf.min():.4f} -> {csf.max():.4f}")
print(f"Prob sum mean   : {prob_sum_in_mask.mean():.4f}")
print(f"Prob sum std    : {prob_sum_in_mask.std():.4f}")
print(f"Prob sum min/max: {prob_sum_in_mask.min():.4f} / {prob_sum_in_mask.max():.4f}")
print(f"Hard labels     : {np.unique(hard).astype(int)}")
print(f"Saved metrics   : {QC_CSV}")
print(f"Saved figure    : {FIG_VOLUME}")
print(f"Saved figure    : {FIG_QC}")
print(f"Saved figure    : {FIG_PORTFOLIO}")

qc_metrics